In [ ]:
!pip install grad-cam

In [ ]:
!pip install grad-cam

import torch
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Target layer (last conv layer of ResNet50)
target_layers = [model.layer4[-1]]

# IMPORTANT: no use_cuda parameter in new version
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)


In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def run_gradcam_pp(img_path, target_class=None):
    """
    img_path: path to chest X-ray
    target_class: int (0 = NORMAL, 1 = PNEUMONIA). If None, uses model prediction.
    """
    # Load image (your helper)
    input_tensor, img_vis = load_image(img_path)  # img_vis may be original size

    # Predict
    pred_class, probs = predict_single(input_tensor)
    if target_class is None:
        target_class = pred_class

    print(f"Target class: {class_names[target_class]} (index {target_class})")
    print("Predicted probs:", probs)

    # Grad-CAM++ targets
    targets = [ClassifierOutputTarget(target_class)]

    # Get CAM (H_cam, W_cam)
    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]  # (H_cam, W_cam)

    # 🔧 Resize img_vis to match CAM size
    H_cam, W_cam = grayscale_cam.shape
    img_pil = Image.fromarray((img_vis * 255).astype(np.uint8))
    img_pil_resized = img_pil.resize((W_cam, H_cam))
    img_vis_resized = np.array(img_pil_resized).astype(np.float32) / 255.0  # (H_cam, W_cam, 3)

    # Overlay heatmap on resized image
    cam_image = show_cam_on_image(img_vis_resized, grayscale_cam, use_rgb=True)

    # Plot
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.title("Original (resized)")
    plt.axis("off")
    plt.imshow(img_vis_resized)

    plt.subplot(1,3,2)
    plt.title("Grad-CAM++")
    plt.axis("off")
    plt.imshow(grayscale_cam, cmap="jet")

    plt.subplot(1,3,3)
    plt.title("Overlay")
    plt.axis("off")
    plt.imshow(cam_image)

    plt.show()
    
run_gradcam_pp("/kaggle/input/chest_xray/test/NORMAL/IM-0001-0001.jpeg")
